# 02. Provider features

Goal: build one table, `provider_features`, with one row per NPI and only the columns a fraud
model should see. Inputs, not verdicts.

## Where the features come from

None of these features exist in the raw files. Every one was chosen and then computed from
raw columns. The choices came from three sources, and it is worth knowing which is which:

1. **Published program-integrity heuristics.** OIG audit reports, CMS comparative billing
   reports, and the public descriptions of CMS's Fraud Prevention System all name the same
   handful of patterns: upcoding office visits, billing the same code repeatedly on one patient
   in one day, intensity per patient that the patient's sickness does not explain, one code
   making up most of the revenue, and schemes aimed at low-income or disabled populations.
   Most of the billing-mix features are those patterns turned into a column.
2. **What the public data allows.** The files are annual totals with no claim dates, no
   diagnoses, and no patient identifiers. That rules out most of what a real program-integrity
   team uses (billing velocity, impossible-day counts, referral networks, patients shared across
   providers). Every feature here had to be computable from yearly totals plus a per-code
   breakdown. This constraint shaped the list more than anything else.
3. **Judgment calls.** A few features are guesses about what might matter, kept because they
   turned out to be predictive or because they let the model account for practice size.
   They are labeled as such below so you can tell them apart from the evidence-backed ones.

Three design rules applied to all of them:

1. **Ratios, not totals.** Raw totals mostly measure practice size. A large honest group beats
   a small fraudulent one on every raw column. Size-independent ratios are what carry signal.
2. **Compare within peers.** Forty services per patient is normal for a dermatologist and
   wild for an internist. Every feature gets a z-score computed inside its provider type.
3. **Leave out what you cannot defend.** Names, addresses, race and sex counts, and the 26
   chronic-condition percentages stay out. The conditions are already summarized by the risk
   score, and 26 noisy columns against 68 positives is how you overfit. Race and sex have no
   defensible role in a fraud score and you would be asked to justify their presence.

Grouping keys (provider type, state, entity code) are kept for comparison and later
drill-down, but they are not model inputs in their raw form.

In [2]:
import sys, time, duckdb
print("python:", sys.executable)
print("duckdb:", duckdb.__version__)
c = duckdb.connect("C:/Users/palla/OneDrive/Documents/Coding Projects/Medicare Fraud ML/database/medicare_fraud.duckdb", read_only=True)
c.execute("SET enable_progress_bar = false")
print("threads:", c.execute("SELECT current_setting('threads')").fetchone()[0])
print("memory_limit:", c.execute("SELECT current_setting('memory_limit')").fetchone()[0])
t = time.time(); c.execute("SELECT count(*) FROM provider").fetchone(); print(f"count provider: {time.time()-t:.2f}s")
t = time.time(); c.execute("SELECT provider_type, median(srvcs_per_bene), mad(srvcs_per_bene) FROM provider_features GROUP BY 1").fetchall(); print(f"median+mad by group: {time.time()-t:.2f}s")
t = time.time(); c.execute("SELECT Rndrng_NPI, count(*), sum(Tot_Srvcs) FROM provider_service GROUP BY 1").fetchall(); print(f"aggregate provider_service: {time.time()-t:.2f}s")
c.close()


python: c:\Users\palla\venvs\medicare-fraud\Scripts\python.exe
duckdb: 1.5.5
threads: 12
memory_limit: 50.0 GiB
count provider: 0.00s
median+mad by group: 0.07s
aggregate provider_service: 0.55s


In [3]:
import duckdb
import pandas as pd

DB_PATH = "C:/Users/palla/OneDrive/Documents/Coding Projects/Medicare Fraud ML/database/medicare_fraud.duckdb"
con = duckdb.connect(DB_PATH)
# DuckDB shows a live progress bar widget in notebooks. VS Code cannot render it and
# the kernel stalls while it tries, so turn it off. Purely cosmetic.
con.execute("SET enable_progress_bar = false")

def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

## 1. Features from the provider table

Source file: `MUP_PHY_R26_P05_V10_D24_Prov.csv`, loaded as the `provider` table. One row per NPI.

Every provider in the file has at least 11 beneficiaries (CMS suppresses anything smaller),
so dividing by `Tot_Benes` is always safe.

| feature | raw columns used | formula | why it matters | source |
|---|---|---|---|---|
| `srvcs_per_bene` | `Tot_Srvcs`, `Tot_Benes` | services / beneficiaries | Intensity per patient. The clearest gap in the raw medians from notebook 01 (5.1 vs 2.9) | heuristic |
| `pymt_per_bene` | `Tot_Mdcr_Pymt_Amt`, `Tot_Benes` | Medicare paid / beneficiaries | Dollar intensity per patient | heuristic |
| `chrg_to_alowd` | `Tot_Sbmtd_Chrg`, `Tot_Mdcr_Alowd_Amt` | submitted charge / allowed amount | My guess was that fraudsters inflate charges. The data showed the opposite: excluded providers bill *closer* to the fee schedule. Kept because it is predictive | judgment |
| `drug_pymt_share` | `Drug_Mdcr_Pymt_Amt`, `Tot_Mdcr_Pymt_Amt` | drug payment / total payment | Infusion and injectable fraud is its own category in OIG enforcement | heuristic |
| `risk_score` | `Bene_Avg_Risk_Scre` | as-is | CMS's own number for how sick the panel is. 1.0 = national average | heuristic |
| `pymt_per_bene_per_risk` | `Tot_Mdcr_Pymt_Amt`, `Tot_Benes`, `Bene_Avg_Risk_Scre` | (paid / beneficiaries) / risk score | Billing not explained by patient sickness. The "healthy panel, heavy billing" mismatch | heuristic |
| `pct_under65` | `Bene_Age_LT_65_Cnt`, `Tot_Benes` | under-65 count / beneficiaries | Under 65 on Medicare means disability or kidney failure. Over-represented in OIG case summaries | heuristic |
| `pct_dual` | `Bene_Dual_Cnt`, `Tot_Benes` | dual-eligible count / beneficiaries | Also on Medicaid, so low income. Frequently targeted | heuristic |
| `avg_age` | `Bene_Avg_Age` | as-is | Panel plausibility | judgment |
| `n_distinct_codes` | `Tot_HCPCS_Cds` | as-is | Breadth of practice. Very narrow or very broad for the specialty is unusual | judgment |
| `tot_benes` | `Tot_Benes` | as-is | Size. Kept so the model can learn that some ratios behave differently at scale | judgment |

**Why the drug columns need care.** CMS suppresses (leaves null) the drug totals for ~150K
providers who gave drugs to fewer than 11 patients. Null here means "small," so
`drug_pymt_share` treats it as zero. The same applies to the age and dual-eligibility counts.
Eight providers were paid nothing at all in 2024, which would make the share 0/0; those get
zero as well.

**What was left on the table.** The 26 chronic-condition percentages, the race and sex counts,
the address and rurality columns, and the geography-adjusted `Stdzd` payment columns. The
standardized payments would be the better dollar measure if the analysis compared across
regions; within a specialty peer group the raw payment is fine and easier to explain.

In [4]:
con.execute("""
CREATE OR REPLACE TABLE feat_provider AS
SELECT
    Rndrng_NPI                                         AS npi,
    Rndrng_Prvdr_Type                                  AS provider_type,
    Rndrng_Prvdr_State_Abrvtn                          AS state,
    Rndrng_Prvdr_Ent_Cd                                AS entity_code,
    Tot_Benes                                          AS tot_benes,
    Tot_HCPCS_Cds                                      AS n_distinct_codes,
    Tot_Srvcs / Tot_Benes                              AS srvcs_per_bene,
    Tot_Mdcr_Pymt_Amt / Tot_Benes                      AS pymt_per_bene,
    Tot_Sbmtd_Chrg / Tot_Mdcr_Alowd_Amt                AS chrg_to_alowd,
    coalesce(coalesce(Drug_Mdcr_Pymt_Amt, 0) / nullif(Tot_Mdcr_Pymt_Amt, 0), 0) AS drug_pymt_share,
    Bene_Avg_Risk_Scre                                 AS risk_score,
    (Tot_Mdcr_Pymt_Amt / Tot_Benes) / Bene_Avg_Risk_Scre AS pymt_per_bene_per_risk,
    coalesce(Bene_Age_LT_65_Cnt, 0) / Tot_Benes        AS pct_under65,
    coalesce(Bene_Dual_Cnt, 0) / Tot_Benes             AS pct_dual,
    Bene_Avg_Age                                       AS avg_age
FROM provider
""")

# Row count, then a look at the first five rows of the new table.
print(q("SELECT count(*) AS n FROM feat_provider"))
q("SELECT * FROM feat_provider LIMIT 5").head()

         n
0  1296739


,npi,provider_type,state,entity_code,tot_benes,n_distinct_codes,srvcs_per_bene,pymt_per_bene,chrg_to_alowd,drug_pymt_share,risk_score,pymt_per_bene_per_risk,pct_under65,pct_dual,avg_age
0,1003000126,Internal Medicine,MD,I,328,15,1.216463,107.699573,4.363510,0.000000,2.5660,41.971774,0.112805,0.234756,76
1,1003000134,Pathology,IL,I,912,21,2.218202,55.019857,4.358538,0.000000,1.1958,46.010919,0.017544,0.052632,76
2,1003000142,Anesthesiology,OH,I,316,41,4.332278,321.449177,2.864839,0.000234,1.6931,189.858353,0.265823,0.354430,70
3,1003000423,Obstetrics & Gynecology,OH,I,75,18,1.866667,95.124267,2.527920,0.000000,0.7931,119.939814,0.000000,0.146667,70
4,1003000480,General Surgery,CO,I,101,28,1.346535,185.809901,4.931953,0.000000,1.4580,127.441633,0.000000,0.336634,68


## 2. Features from the provider_service table

Source file: `PHY_R26_P05_V10_D24_Prov_Svc.csv`, loaded as the `provider_service` table. One
row per NPI x procedure code x place of service, 9.8M rows.

Everything in the provider table is a yearly total. This table is the only place the *mix* of
what a provider bills is visible, and billing mix is where most published fraud patterns live.
The table is aggregated down to one row per NPI in SQL; it never enters pandas.

| feature | raw columns used | formula | why it matters | source |
|---|---|---|---|---|
| `sameday_repeat_ratio` | `Tot_Srvcs`, `Tot_Bene_Day_Srvcs`, `HCPCS_Drug_Ind` | sum of services / sum of patient-day services, non-drug rows only | Above 1.0 means the same code was billed more than once per patient per day. Duplicate billing and unbundling. CMS publishes `Tot_Bene_Day_Srvcs` specifically so this can be computed | heuristic |
| `em_high_share` | `HCPCS_Cd`, `Tot_Srvcs` | services on 99214 + 99215 / services on 99212 through 99215 | Upcoding office visits. OIG has audited this repeatedly. Null for providers who bill no office visits | heuristic |
| `top_code_pymt_share` | `Tot_Srvcs`, `Avg_Mdcr_Pymt_Amt` | max over codes of (services x avg payment) / sum over codes | One code making up most of the revenue. Fraud rings often run one code at volume | heuristic |
| `facility_share` | `Place_Of_Srvc`, `Tot_Srvcs` | services with place F / all services | Place-of-service manipulation. The same code pays differently in a facility than an office | heuristic |
| `n_service_rows` | count of rows | count | Breadth, finer than distinct codes because it also splits by place | judgment |

**Two traps in the raw data.** Drug rows count dose units as "services," so a single infusion
can be 400 services on one patient-day. The same-day ratio is computed on non-drug rows only,
using the `HCPCS_Drug_Ind` flag. And 99211 (the lowest office-visit level) was effectively
retired in 2021, so the E&M denominator starts at 99212.

**Coverage.** Only 1.21M of 1.30M providers appear in this table at all. The other ~89K had
every code-level row suppressed (fewer than 11 patients on every code). Their service features
are null and the models must tolerate that.

In [5]:
con.execute("""
CREATE OR REPLACE TABLE feat_service AS
SELECT
    Rndrng_NPI AS npi,
    count(*)   AS n_service_rows,

    sum(CASE WHEN HCPCS_Drug_Ind = 'N' THEN Tot_Srvcs END)
      / nullif(sum(CASE WHEN HCPCS_Drug_Ind = 'N' THEN Tot_Bene_Day_Srvcs END), 0)
      AS sameday_repeat_ratio,

    sum(CASE WHEN HCPCS_Cd IN ('99214','99215') THEN Tot_Srvcs END)
      / nullif(sum(CASE WHEN HCPCS_Cd IN ('99212','99213','99214','99215') THEN Tot_Srvcs END), 0)
      AS em_high_share,

    max(Tot_Srvcs * Avg_Mdcr_Pymt_Amt) / nullif(sum(Tot_Srvcs * Avg_Mdcr_Pymt_Amt), 0)
      AS top_code_pymt_share,

    coalesce(sum(CASE WHEN Place_Of_Srvc = 'F' THEN Tot_Srvcs END), 0) / sum(Tot_Srvcs)
      AS facility_share
FROM provider_service
GROUP BY Rndrng_NPI
""")

# Row count and how many providers have no office-visit codes, then the first five rows.
print(q("SELECT count(*) AS n, sum(em_high_share IS NULL) AS em_null FROM feat_service"))
q("SELECT * FROM feat_service LIMIT 5").head()

         n   em_null
0  1207473  695975.0


,npi,n_service_rows,sameday_repeat_ratio,em_high_share,top_code_pymt_share,facility_share
0,1124101597,7,1.000000,0.850746,0.478952,0.0
1,1124105341,2,1.000000,0.725543,0.852491,0.0
2,1124105390,15,1.000881,0.107018,0.452293,0.0
3,1124107255,1,1.000000,1.000000,1.000000,0.0
4,1124111273,8,1.000000,0.673307,0.437214,0.0


## 3. Peer z-scores

Source: not a file. This is a transformation applied to every feature above, and the method is
standard robust statistics, not something specific to fraud.

A z-score says how many spreads a value sits from the center of its group. The ordinary
version uses mean and standard deviation, but billing data has heavy right tails: a handful of
providers billing millions would drag the mean and inflate the standard deviation for everyone.

So we use the **robust** version. Center is the median. Spread is the median absolute deviation
(MAD), scaled by 1.4826 so that it equals the standard deviation when the data happen to be normal.

```
z = (x - median_group) / (1.4826 * MAD_group)
```

**Where the choices came from:**

- **Median and MAD instead of mean and SD**: textbook robust statistics. Any outlier-heavy
  dataset gets this treatment.
- **1.4826**: a mathematical constant. For a normal distribution MAD is 0.6745 of the standard
  deviation, and 1 / 0.6745 = 1.4826. Multiplying by it puts MAD on the same scale as SD so
  "z of 3 is unusual" keeps its textbook meaning.
- **Groups are provider types**: because `Rndrng_Prvdr_Type` is the only peer-group key the
  file offers. Specialty is what CMS's own comparative billing reports group by.
- **Minimum group size of 100**: a judgment call. Below that, a median and MAD from a
  handful of providers are unstable. 20 of the 113 types fall under it and use global statistics.
- **The fallback chain**: MAD is zero when more than half a group shares one value (drug share
  is zero for most, facility share is exactly 1.0 for most), and you cannot divide by zero. For
  those, spread falls back to the ordinary standard deviation, which is never zero unless the
  whole column is constant. Order: group MAD, then global MAD, then group SD, then global SD.

The SQL below is generated from a list of feature names rather than written 16 times by hand.
Print `sql` if you want to read the expanded version.

In [6]:
FEATURES = [
    "srvcs_per_bene", "pymt_per_bene", "chrg_to_alowd", "drug_pymt_share",
    "risk_score", "pymt_per_bene_per_risk", "pct_under65", "pct_dual", "avg_age",
    "n_distinct_codes", "tot_benes",
    "sameday_repeat_ratio", "em_high_share", "top_code_pymt_share", "facility_share",
    "n_service_rows",
]
MIN_GROUP = 100
K = 1.4826

stat_cols = ",\n    ".join(
    f"median({f}) AS med_{f}, mad({f}) AS mad_{f}, stddev_samp({f}::DOUBLE) AS sd_{f}"
    for f in FEATURES
)
z_cols = ",\n    ".join(
    f"(b.{f} - coalesce(g.med_{f}, a.med_{f})) / coalesce("
    f"nullif({K} * g.mad_{f}, 0), nullif({K} * a.mad_{f}, 0), "
    f"nullif(g.sd_{f}, 0), nullif(a.sd_{f}, 0)) AS z_{f}"
    for f in FEATURES
)
raw_cols = ",\n    ".join(f"b.{f}" for f in FEATURES)

sql = f"""
CREATE OR REPLACE TABLE provider_features AS
WITH base AS (
    SELECT p.*, s.n_service_rows, s.sameday_repeat_ratio, s.em_high_share,
           s.top_code_pymt_share, s.facility_share,
           lb.label
    FROM feat_provider p
    LEFT JOIN feat_service s USING (npi)
    LEFT JOIN labels lb USING (npi)
),
grp AS (
    SELECT provider_type, count(*) AS n_in_group,
    {stat_cols}
    FROM base GROUP BY provider_type
    HAVING count(*) >= {MIN_GROUP}
),
alls AS (
    SELECT {stat_cols}
    FROM base
)
SELECT
    b.npi, b.provider_type, b.state, b.entity_code, b.label,
    coalesce(g.n_in_group, 0) >= {MIN_GROUP} AS has_peer_group,
    {raw_cols},
    {z_cols}
FROM base b
LEFT JOIN grp g USING (provider_type)
CROSS JOIN alls a
"""
con.execute(sql)
q("SELECT count(*) AS n, sum(has_peer_group) AS with_peers, count(*) - sum(has_peer_group) AS global_fallback FROM provider_features")

,n,with_peers,global_fallback
0,1296739,1296259.0,480.0


## 4. Flag extreme providers

Source: this is the rule-based approach that program-integrity work used before machine
learning, and it is here as a baseline. Any model built later has to beat it.

A z-score of 3 or more means "three spreads away from normal for people like you." Counting how
many features a provider is extreme on gives the simplest possible fraud detector: a rule.

Two columns are added to `provider_features`. Nobody is removed from the table. The models
need to see the extreme providers alongside the ordinary ones.

| column | built from | meaning |
|---|---|---|
| `n_extreme` | all 16 `z_*` columns | how many have absolute value 3 or more |
| `max_abs_z` | all 16 `z_*` columns | the single largest absolute z-score |

**Why 3:** convention. Under a normal distribution about 0.3% of values fall beyond 3
spreads, which is the usual meaning of "rare." Billing data is not normal, so far more than
0.3% land there, and the table below shows how many.

**Why absolute value:** excluded providers sit *below* their peers on charge-to-allowed, so a
large negative z is as much a flag as a large positive one.

**What the next cell does:** rebuilds `provider_features` with the two new columns appended,
then shows how many providers have zero, one, two, or more extreme features.

In [7]:
Z_CUT = 3.0

n_extreme_expr = " + ".join(f"CASE WHEN abs(z_{f}) >= {Z_CUT} THEN 1 ELSE 0 END" for f in FEATURES)
max_abs_expr   = "greatest(" + ", ".join(f"coalesce(abs(z_{f}), 0)" for f in FEATURES) + ")"

con.execute(f"""
CREATE OR REPLACE TABLE provider_features AS
SELECT *,
       {n_extreme_expr} AS n_extreme,
       {max_abs_expr}   AS max_abs_z
FROM provider_features
""")

q("""
SELECT n_extreme, count(*) AS n_providers,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM provider_features
GROUP BY 1 ORDER BY 1
""")

,n_extreme,n_providers,pct
0,0,742127,57.23
1,1,300273,23.16
2,2,118952,9.17
3,3,71254,5.49
4,4,36173,2.79
5,5,17947,1.38
6,6,7160,0.55
7,7,2248,0.17
8,8,510,0.04
9,9,80,0.01


**What the next cell does:** asks the question that matters. Of the 68 known positives, how many
would the rule catch at each cutoff on `n_extreme`? And what share of the whole population would
be flagged alongside them?

**Why:** this is the rule's precision and recall. If most positives have at least one extreme
feature, the rule already works and the models must do better. If most do not, the caught
providers are not extreme on any single feature and only the combination separates them, which
is the case for a model. Either answer is useful.

In [8]:
q("""
WITH cuts AS (SELECT unnest([1, 2, 3, 4]) AS min_extreme)
SELECT c.min_extreme,
       sum(CASE WHEN p.n_extreme >= c.min_extreme THEN 1 ELSE 0 END)                    AS n_flagged,
       sum(CASE WHEN p.n_extreme >= c.min_extreme AND p.label = 1 THEN 1 ELSE 0 END)    AS positives_caught,
       round(100.0 * sum(CASE WHEN p.n_extreme >= c.min_extreme AND p.label = 1 THEN 1 ELSE 0 END)
             / sum(CASE WHEN p.label = 1 THEN 1 ELSE 0 END), 1)                          AS pct_positives_caught,
       round(100.0 * sum(CASE WHEN p.n_extreme >= c.min_extreme THEN 1 ELSE 0 END)
             / count(*), 2)                                                              AS pct_population_flagged
FROM provider_features p
CROSS JOIN cuts c
WHERE p.label IS NOT NULL
GROUP BY 1 ORDER BY 1
""")

,min_extreme,n_flagged,positives_caught,pct_positives_caught,pct_population_flagged
0,1,554569.0,38.0,55.9,42.77
1,2,254313.0,19.0,27.9,19.61
2,3,135372.0,11.0,16.2,10.44
3,4,64123.0,10.0,14.7,4.95


## 5. Does the table look right?

Three checks. Nulls per column, so we know what the model has to tolerate. Then the raw medians
by label, as in the load notebook but now on the engineered features. Then the z-score medians
by label, which is the real test: after peer adjustment, do the positives still stand out?

A z-score median near 0 for the negatives is expected by construction. What we want to see is
the positives' median pushed away from 0 on at least a few features. The `gap` column is the
positives' median minus the negatives' median, sorted by size, and it is the first rough
ranking of which features carry signal. Notebook 05's importance chart is the model's version
of the same question.

In [9]:
null_counts = q(f"""
SELECT {", ".join(f"sum({f} IS NULL) AS {f}" for f in FEATURES)}
FROM provider_features
""").T
null_counts.columns = ["n_null"]
null_counts

,n_null
srvcs_per_bene,0.0
pymt_per_bene,0.0
chrg_to_alowd,0.0
drug_pymt_share,0.0
risk_score,0.0
pymt_per_bene_per_risk,0.0
pct_under65,0.0
pct_dual,0.0
avg_age,0.0
n_distinct_codes,0.0


In [10]:
raw_by_label = q(f"""
SELECT label, count(*) AS n,
       {", ".join(f"round(median({f}), 3) AS {f}" for f in FEATURES)}
FROM provider_features
WHERE label IS NOT NULL
GROUP BY label ORDER BY label
""").set_index("label").T
raw_by_label

label,0,1
n,1296590.000,68.000
srvcs_per_bene,2.873,5.121
pymt_per_bene,187.848,269.459
chrg_to_alowd,2.851,1.876
drug_pymt_share,0.000,0.000
risk_score,1.355,1.301
pymt_per_bene_per_risk,126.870,191.034
pct_under65,0.000,0.000
pct_dual,0.080,0.107
avg_age,74.000,73.000


In [11]:
z_by_label = q(f"""
SELECT label,
       {", ".join(f"round(median(z_{f}), 2) AS z_{f}" for f in FEATURES)}
FROM provider_features
WHERE label IS NOT NULL
GROUP BY label ORDER BY label
""").set_index("label").T
z_by_label["gap"] = z_by_label[1] - z_by_label[0]
z_by_label.sort_values("gap", key=abs, ascending=False)

label,0,1,gap
z_srvcs_per_bene,0.0,0.56,0.56
z_chrg_to_alowd,0.0,-0.53,-0.53
z_em_high_share,0.0,0.52,0.52
z_pymt_per_bene_per_risk,0.0,0.27,0.27
z_n_distinct_codes,0.0,-0.25,-0.25
z_risk_score,0.0,-0.09,-0.09
z_tot_benes,0.0,-0.07,-0.07
z_pct_dual,0.0,0.04,0.04
z_top_code_pymt_share,0.0,-0.04,-0.04
z_pymt_per_bene,0.0,0.02,0.02


## 6. Clean up

The two intermediate tables are dropped. `provider_features` is the only thing later notebooks read.

In [12]:
con.execute("DROP TABLE feat_provider")
con.execute("DROP TABLE feat_service")
q("SELECT table_name, estimated_size AS approx_rows FROM duckdb_tables() ORDER BY 1")

,table_name,approx_rows
0,labels,1296739
1,leie,83842
2,provider,1296739
3,provider_features,1296739
4,provider_service,9781673


In [13]:
con.close()